# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Feb_2026_test_1_ac",
}

In [3]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 100.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 1,  # number of aircraft
}

In [4]:
# plume dispersion parameters
pl_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "n_slices": 3,  # number of slices in the plume
    "f_max": 0.99,  # maximum fraction of total emissions in any slice
    "shear": 0.01,  # shear [m/s]
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 3.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO", "CO", "SO2"),
    # "species_pl": ("NO", "CO", "SO2"),
    "species_pl": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
pl_params = PlParams(**pl_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, pl_params, met_params, chem_params)

In [8]:
gpat.preprocess_gpat()

/home/ktait98/miniconda3/envs/contrails/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:234: UserWarning: no explicit representation of timezones available for np.datetime64
  return data.astype(dtype, **kwargs)


flight 0 done


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:618: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i].dataframe[column] = fl[i].dataframe[column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:681: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:898: RuntimeWarning: invalid value encountered in cast
  age_seconds = np.where(np.isnat(age_values), 0, (age_values / np.timedelta64(1, 's')).astype(int))


Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_1_ac/boxm_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_1_ac/fl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Feb_2026_test_1_ac/pl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_1_ac/boxm_out.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_1_ac/patch_table.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Feb_2026_test_1_ac/pl_out.nc


In [9]:
gpat.eval()

 FL_DS SUMMARY:
   NSEG =           23
   IS_OPEN =  T
   NCID =        65536
 PL_DS SUMMARY:
   NSEG =           23
   NSEMI =            3
   NTPL =          134
   IS_OPEN =  T
   NCID =       131072
 BOXM_DS SUMMARY:
   NCELL =          800
   NSBOXM =          219
   NTBOXM =          721
   IS_OPEN =  T
   NCID =       196608
 PL_OUT_WRITE: SPECIES_ID/PL_NUM =           1           8  sum/min/max =   1.0837914650478422        0.0000000000000000        1.0837914650478422     
 PL_OUT_WRITE: SPECIES_ID/PL_NUM =           2           4  sum/min/max =   0.0000000000000000        0.0000000000000000        0.0000000000000000     
 PL_OUT_WRITE: SPECIES_ID/PL_NUM =           3           6  sum/min/max =   0.0000000000000000        0.0000000000000000        0.0000000000000000     
 PL_OUT_WRITE: SPECIES_ID/PL_NUM =           4           5  sum/min/max =   0.0000000000000000        0.0000000000000000        0.0000000000000000     
 PL_OUT_WRITE: SPECIES_ID/PL_NUM =           5           7

In [10]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")
fl_ds

<xarray.Dataset> Size: 5kB
Dimensions:               (seg_id: 23)
Coordinates:
    flight_id             (seg_id) int64 184B ...
    waypoint              (seg_id) int64 184B ...
  * seg_id                (seg_id) int64 184B 1 2 3 4 5 6 ... 18 19 20 21 22 23
Data variables: (12/16)
    longitude             (seg_id) float64 184B ...
    latitude              (seg_id) float64 184B ...
    level                 (seg_id) float64 184B ...
    altitude              (seg_id) float64 184B ...
    time                  (seg_id) <U20 2kB ...
    true_airspeed         (seg_id) float64 184B ...
    ...                    ...
    thrust                (seg_id) float64 184B ...
    rocd                  (seg_id) float64 184B ...
    fuel_flow_per_engine  (seg_id) float64 184B ...
    thrust_setting        (seg_id) float64 184B ...
    time_rel_s            (seg_id) int64 184B ...
    time_idx              (seg_id) int64 184B ...
Attributes:
    description:  Flight trajectory and emissions data for BOXM

In [11]:
from IPython.display import clear_output
clear_output(wait=True)

pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds

<xarray.Dataset> Size: 791kB
Dimensions:          (seg_id: 23, time: 134, ht: 2, species_emi: 3)
Coordinates:
    flight_id        (seg_id) int64 184B ...
    waypoint         (seg_id) int64 184B ...
  * seg_id           (seg_id) int64 184B 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23
  * time             (time) <U20 11kB '2022-01-20T13:00:00Z' ... '2022-01-20T...
    species_emi_num  (species_emi) int64 24B ...
  * species_emi      (species_emi) <U3 36B 'NO' 'CO' 'SO2'
    time_rel_s       (time) int64 1kB ...
    time_idx         (time) int64 1kB ...
  * ht               (ht) <U4 32B 'tail' 'head'
Data variables: (12/13)
    age              (seg_id, time) <U21 259kB ...
    longitude        (seg_id, ht, time) float64 49kB ...
    latitude         (seg_id, ht, time) float64 49kB ...
    level            (seg_id, ht, time) float64 49kB ...
    width            (seg_id, ht, time) float64 49kB ...
    depth            (seg_id, ht, time) float64 49kB ...
    ...               ...
    sigma_yy         (seg_id, ht, time) float64 49kB ...
    sigma_yz         (seg_id, ht, time) float64 49kB ...
    sigma_zz         (seg_id, ht, time) float64 49kB ...
    altitude         (seg_id, ht, time) float64 49kB ...
    emi_pl_mass      (seg_id, species_emi) float64 552B ...
    age_s            (seg_id, time) int64 25kB ...
Attributes:
    nseg:             23
    ts_fl:            60.0
    ts_pl:            60.0
    ts_sim:           20.0
    ts_out:           60.0
    species_emi:      ['NO', 'CO', 'SO2']
    species_pl:       ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'HO2...
    species_emi_num:  [ 8 11 16]
    species_pl_num:   [  8   4   6   5   7  14  13  15 198 217  12 144  11  2...
    n_slices:         3
    f_max:            0.99
    description:      Emission species mass in plume segments

In [12]:
from IPython.display import clear_output
clear_output(wait=True)

boxm_out = xr.open_dataset(f"{gpat.outputs_job}/boxm_out.nc")

boxm_out["time_idx"].values

array([  1,   4,   7,  10,  13,  16,  19,  22,  25,  28,  31,  34,  37,
        40,  43,  46,  49,  52,  55,  58,  61,  64,  67,  70,  73,  76,
        79,  82,  85,  88,  91,  94,  97, 100, 103, 106, 109, 112, 115,
       118, 121, 124, 127, 130, 133, 136, 139, 142, 145, 148, 151, 154,
       157, 160, 163, 166, 169, 172, 175, 178, 181, 184, 187, 190, 193,
       196, 199, 202, 205, 208, 211, 214, 217, 220, 223, 226, 229, 232,
       235, 238, 241, 244, 247, 250, 253, 256, 259, 262, 265, 268, 271,
       274, 277, 280, 283, 286, 289, 292, 295, 298, 301, 304, 307, 310,
       313, 316, 319, 322, 325, 328, 331, 334, 337, 340, 343, 346, 349,
       352, 355, 358, 361, 364, 367, 370, 373, 376, 379, 382, 385, 388,
       391, 394, 397, 400, 403, 406, 409, 412, 415, 418, 421, 424, 427,
       430, 433, 436, 439, 442, 445, 448, 451, 454, 457, 460, 463, 466,
       469, 472, 475, 478, 481, 484, 487, 490, 493, 496, 499, 502, 505,
       508, 511, 514, 517, 520, 523, 526, 529, 532, 535, 538, 54

In [13]:
gpat.patch_table

<xarray.Dataset> Size: 512B
Dimensions:          (row: 0, species_out: 16)
Coordinates:
  * row              (row) int64 0B 
    patch_id         (row) int64 0B 
  * species_out      (species_out) <U6 384B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    species_out_num  (species_out) int64 128B 6 4 8 5 7 14 ... 21 22 15 198 16
    time             (row) object 0B 
    time_rel_s       (row) int64 0B 
    time_idx         (row) int64 0B 
    latitude_f       (row) float64 0B 
    longitude_f      (row) float64 0B 
    altitude_f       (row) float64 0B 
    level_f          (row) float64 0B 
Data variables:
    Y_del_f          (row, species_out) float64 0B dask.array<chunksize=(0, 16), meta=np.ndarray>
Attributes:
    description:  Fine grid plume patch output for BOXM

In [14]:
from IPython.display import clear_output
clear_output(wait=True)

pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out.seg_id.values


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23])

In [15]:
from IPython.display import clear_output
clear_output(wait=True)

np.set_printoptions(threshold=np.inf, linewidth=2000)
print(pl_out["pl_mass"].isel(time=slice(60, 80)).values)

[[[1.08379147 0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
  [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
  [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
  [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.         0.        ]
  [0.         0.         0.         0.         0.         0.         0.         0.         0.         0.

In [16]:
no = pl_out["pl_mass"].sel(species_pl="NO")

# Show which time_idx are nonzero for the first segment
seg0 = no.isel(seg_id=0).values
nonzero_time_idx = pl_out["time_idx"].values[seg0 != 0]

print(nonzero_time_idx[:10])

[181]


In [17]:
from IPython.display import clear_output
clear_output(wait=True)

# Sanity: CO slice only
co = pl_out["pl_mass"].sel(species_pl="CO")
no = pl_out["pl_mass"].sel(species_pl="NO")

print("CO max:", float(co.max()))
print("NO max:", float(no.max()))

# Where the nonzeros actually are
print("CO nonzero count:", int((co.values != 0).sum()))
print("NO nonzero count:", int((no.values != 0).sum()))

CO max: 1.0837914650478422
NO max: 1.0837914650478422
CO nonzero count: 23
NO nonzero count: 23


In [18]:
pl_out["pl_mass"].max(("seg_id","time")).to_pandas()

species_pl
NO          1.083791
NO2         0.000000
O3          0.000000
NO3         0.000000
N2O5        0.000000
HNO3        0.000000
HONO        0.000000
HO2NO2      0.000000
PAN         0.000000
CH3O2NO2    0.000000
H2O2        0.000000
CH3OOH      0.000000
CO          1.083791
CH4         0.000000
HCHO        0.000000
SO2         1.083791
SA          0.000000
Name: pl_mass, dtype: float64